# Обучение агрономических моделей (RandomForest)

В этом ноутбуке обучаются четыре классификационные модели на основе RandomForest:

| Модель | Датасет | Задача |
|--------|---------|--------|
| **Crop Recommender** | `crop.csv` | Рекомендация культуры по почве и климату |
| **Fertilizer Recommender** | `fertilizer.csv` | Рекомендация удобрения по культуре, почве и дефициту |
| **Pesticide Recommender** | `pesticide.csv` | Рекомендация пестицида по культуре, вредителю и интенсивности |
| **Disease Predictor** | `disease.csv` | Прогноз болезни и уровня риска по погоде и стадии роста |

**Валидация:** Stratified 5-Fold CV по метрике F1-macro  
**Артефакты сохраняются в:** `agro-ml-service/models/`

---

## Структура ноутбука
1. Настройка путей и общие параметры
2. Модель 1: Рекомендация культуры (crop)
3. Модель 2: Рекомендация удобрения (fertilizer)
4. Модель 3: Рекомендация пестицида (pesticide)
5. Модель 4: Прогноз болезни (disease)
6. Итоги и список артефактов

## 1. Настройка путей и общие параметры

In [ ]:
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

warnings.filterwarnings('ignore')

# Ноутбук лежит в agro-ml-service/train/
# Датасеты — в analytics/data/ (на два уровня выше)
NOTEBOOK_DIR = Path('.').resolve()
SERVICE_DIR  = NOTEBOOK_DIR.parent          # agro-ml-service/
PROJECT_ROOT = SERVICE_DIR.parent           # AgroPlanPro/
DATA_DIR     = PROJECT_ROOT / 'analytics' / 'data'
MODELS_DIR   = SERVICE_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print(f'Датасеты : {DATA_DIR}')
print(f'Модели   : {MODELS_DIR}')

# Общие параметры RandomForest для всех моделей
RF_PARAMS = dict(
    n_estimators=200,
    max_depth=None,       # деревья растут до полной чистоты (контролируется min_samples_leaf)
    min_samples_leaf=2,   # минимум 2 образца в листе — защита от переобучения
    random_state=42,
    n_jobs=-1,
)

# Stratified 5-Fold — сохраняет распределение классов в каждом фолде
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_cv(model, X, y):
    """Запускает CV, печатает результат, затем обучает модель на всех данных."""
    scores = cross_val_score(model, X, y, cv=CV, scoring='f1_macro')
    print(f'  CV F1-macro (5-fold): {scores.mean():.4f} +/- {scores.std():.4f}')
    model.fit(X, y)   # финальное обучение на всех данных
    is_acc = accuracy_score(y, model.predict(X))
    print(f'  In-sample accuracy  : {is_acc:.4f}  (справочно, не метрика качества)')
    return model

print('Конфигурация готова')

## 2. Модель 1: Рекомендация культуры

**Датасет:** `crop.csv`  
**Признаки:** N, P, K (питательные вещества почвы), temperature, humidity, ph, rainfall  
**Таргет:** `label` — название культуры  
**Данные:** синтетический датасет, сгенерированный по агрономическим правилам для Западной Сибири

Модель даёт вероятности по каждой культуре — сервис возвращает топ-3 с confidence.

In [ ]:
crop_csv = DATA_DIR / 'crop.csv'
df_crop  = pd.read_csv(crop_csv)

print(f'Строк: {len(df_crop):,} | Столбцы: {list(df_crop.columns)}')
print(f'Культур: {df_crop["label"].nunique()} — {sorted(df_crop["label"].unique())}')
df_crop.head()

In [ ]:
# Распределение классов
class_counts = df_crop['label'].value_counts()
fig, ax = plt.subplots(figsize=(10, 3))
class_counts.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Распределение классов (crop.csv)')
ax.set_xlabel('Культура')
ax.set_ylabel('Количество записей')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Баланс классов:')
print(class_counts.to_string())

In [ ]:
# Корреляционная матрица числовых признаков
feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
target_col   = 'label'

fig, ax = plt.subplots(figsize=(7, 5))
corr = df_crop[feature_cols].corr()
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right')
ax.set_yticklabels(feature_cols)
ax.set_title('Корреляция признаков (crop)')
plt.tight_layout()
plt.show()

In [ ]:
X_crop = df_crop[feature_cols].values

# LabelEncoder переводит строковые метки классов в целые числа
le_crop = LabelEncoder()
y_crop  = le_crop.fit_transform(df_crop[target_col].astype(str))

print(f'Классы ({len(le_crop.classes_)}): {list(le_crop.classes_)}')
print(f'Форма X: {X_crop.shape}')

model_crop = RandomForestClassifier(**RF_PARAMS)
model_crop = run_cv(model_crop, X_crop, y_crop)

In [ ]:
# Полный classification report по in-sample данным
print(classification_report(
    y_crop, model_crop.predict(X_crop),
    target_names=le_crop.classes_
))

In [ ]:
# Feature importance для модели рекомендации культуры
crop_imp = pd.Series(
    model_crop.feature_importances_, index=feature_cols
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
crop_imp.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Важность признаков (crop model)')
ax.set_xlabel('Feature importance (Gini)')
plt.tight_layout()
plt.show()

In [ ]:
# Сохранение
joblib.dump(model_crop, MODELS_DIR / 'crop_model.pkl')
joblib.dump(le_crop,    MODELS_DIR / 'crop_label_encoder.pkl')
print('Сохранено: crop_model.pkl, crop_label_encoder.pkl')

## 3. Модель 2: Рекомендация удобрения

**Датасет:** `fertilizer.csv`  
**Признаки:** crop_type, soil_type, deficiency_level (все категориальные — label-encoded)  
**Таргет:** название удобрения  

Особенность: soil_type из запроса автоматически маппится к трём типам модели
(loamy / chernozem / solonetz) через `_SOIL_TEXTURE_MAP` в `agro_predictor.py`.

In [ ]:
fert_csv = DATA_DIR / 'fertilizer.csv'
df_fert  = pd.read_csv(fert_csv)

print(f'Строк: {len(df_fert):,} | Столбцы: {list(df_fert.columns)}')
df_fert.head()

In [ ]:
# Гибкий маппинг имён столбцов — датасет может иметь разные названия
col_map = {}
for src, candidates in [
    ('crop_type',        ['crop_type', 'Crop Type', 'crop']),
    ('soil_type',        ['soil_type', 'Soil Type', 'soil']),
    ('deficiency_level', ['deficiency_level', 'Deficiency Level', 'deficiency']),
    ('fertilizer',       ['fertilizer', 'Fertilizer Name', 'Fertilizer']),
]:
    for c in candidates:
        if c in df_fert.columns:
            col_map[src] = c
            break

print('Маппинг столбцов:', col_map)

# Уникальные значения в каждом признаке
for key, col in col_map.items():
    print(f'  {key}: {sorted(df_fert[col].unique())}')

In [ ]:
fert_encoders = {}
X_fert_parts  = []

# Кодируем каждый категориальный признак отдельным LabelEncoder
for col in ['crop_type', 'soil_type', 'deficiency_level']:
    le = LabelEncoder()
    le.fit(df_fert[col_map[col]].astype(str))
    fert_encoders[col] = le
    X_fert_parts.append(le.transform(df_fert[col_map[col]].astype(str)).reshape(-1, 1))

# Целевой энкодер — название удобрения
le_fert = LabelEncoder()
y_fert  = le_fert.fit_transform(df_fert[col_map['fertilizer']].astype(str))
fert_encoders['fertilizer'] = le_fert

X_fert = np.hstack(X_fert_parts)

print(f'Удобрений ({len(le_fert.classes_)}): {list(le_fert.classes_)}')
print(f'Форма X: {X_fert.shape}')

model_fert = RandomForestClassifier(**RF_PARAMS)
model_fert = run_cv(model_fert, X_fert, y_fert)

In [ ]:
# Матрица ошибок для наглядности
cm = confusion_matrix(y_fert, model_fert.predict(X_fert))
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=le_fert.classes_)
disp.plot(ax=ax, colorbar=True, xticks_rotation='vertical')
ax.set_title('Матрица ошибок (fertilizer model, in-sample)')
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(model_fert,    MODELS_DIR / 'fertilizer_model.pkl')
joblib.dump(fert_encoders, MODELS_DIR / 'fertilizer_encoders.pkl')
print('Сохранено: fertilizer_model.pkl, fertilizer_encoders.pkl')

## 4. Модель 3: Рекомендация пестицида

**Датасет:** `pesticide.csv`  
**Признаки:** crop, pest_type, intensity  
**Таргет:** название пестицида  

Для каждого предсказанного пестицида сервис дополнительно достаёт
действующее вещество из словаря `_ACTIVE_INGREDIENTS` в `agro_predictor.py`.

In [ ]:
pest_csv = DATA_DIR / 'pesticide.csv'
df_pest  = pd.read_csv(pest_csv)

print(f'Строк: {len(df_pest):,} | Столбцы: {list(df_pest.columns)}')
df_pest.head()

In [ ]:
col_map_pest = {}
for src, candidates in [
    ('crop',      ['crop', 'Crop', 'crop_type']),
    ('pest_type', ['pest_type', 'Pest Type', 'pest']),
    ('intensity', ['intensity', 'Intensity', 'severity']),
    ('pesticide', ['pesticide', 'Pesticide', 'Pesticide Name']),
]:
    for c in candidates:
        if c in df_pest.columns:
            col_map_pest[src] = c
            break

print('Маппинг:', col_map_pest)
for key, col in col_map_pest.items():
    print(f'  {key}: {sorted(df_pest[col].unique())}')

In [ ]:
pest_encoders = {}
X_pest_parts  = []

for col in ['crop', 'pest_type', 'intensity']:
    le = LabelEncoder()
    le.fit(df_pest[col_map_pest[col]].astype(str))
    pest_encoders[col] = le
    X_pest_parts.append(le.transform(df_pest[col_map_pest[col]].astype(str)).reshape(-1, 1))

le_pesticide = LabelEncoder()
y_pest       = le_pesticide.fit_transform(df_pest[col_map_pest['pesticide']].astype(str))
pest_encoders['pesticide'] = le_pesticide

X_pest = np.hstack(X_pest_parts)

print(f'Пестицидов ({len(le_pesticide.classes_)}): {list(le_pesticide.classes_)}')
print(f'Форма X: {X_pest.shape}')

model_pest = RandomForestClassifier(**RF_PARAMS)
model_pest = run_cv(model_pest, X_pest, y_pest)

In [ ]:
print(classification_report(
    y_pest, model_pest.predict(X_pest),
    target_names=le_pesticide.classes_
))

In [ ]:
joblib.dump(model_pest,    MODELS_DIR / 'pesticide_model.pkl')
joblib.dump(pest_encoders, MODELS_DIR / 'pesticide_encoders.pkl')
print('Сохранено: pesticide_model.pkl, pesticide_encoders.pkl')

## 5. Модель 4: Прогноз болезни растений

**Датасет:** `disease.csv`  
**Признаки:** crop (cat), growth_stage (cat), temperature, humidity, rainfall (num)  
**Таргеты (два):**
- `disease` — название болезни (DiseaseClassifier)
- `risk_level` — уровень риска: low / medium / high (RiskClassifier)

Обе модели обучаются на одних и тех же признаках X, но с разными таргетами.
Сервис вызывает их оба в одном запросе и возвращает результат совместно.

In [ ]:
disease_csv = DATA_DIR / 'disease.csv'
df_disease  = pd.read_csv(disease_csv)

print(f'Строк: {len(df_disease):,} | Столбцы: {list(df_disease.columns)}')
print(f'Болезней: {df_disease["disease"].nunique()} — {sorted(df_disease["disease"].unique())}')
print(f'Уровни риска: {sorted(df_disease["risk_level"].unique())}')
df_disease.head()

In [ ]:
# Распределение болезней в датасете
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_disease['disease'].value_counts().plot(
    kind='bar', ax=axes[0], color='steelblue', alpha=0.8
)
axes[0].set_title('Распределение болезней')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

df_disease['risk_level'].value_counts().plot(
    kind='bar', ax=axes[1], color='coral', alpha=0.8
)
axes[1].set_title('Распределение уровней риска')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
feature_cat = ['crop', 'growth_stage']
feature_num = ['temperature', 'humidity', 'rainfall']

disease_encoders = {}
X_disease_parts  = []

# Кодируем категориальные признаки
for col in feature_cat:
    le = LabelEncoder()
    le.fit(df_disease[col].astype(str))
    disease_encoders[col] = le
    X_disease_parts.append(le.transform(df_disease[col].astype(str)).reshape(-1, 1))
    print(f'  {col}: {list(le.classes_)}')

# Числовые признаки добавляем напрямую
X_disease_numeric = df_disease[feature_num].values
X_disease = np.hstack(X_disease_parts + [X_disease_numeric])

print(f'\nФорма X: {X_disease.shape}')
print(f'Порядок столбцов: {feature_cat} + {feature_num}')

In [ ]:
# --- Classifier 1: болезнь ---
print('=== DiseaseClassifier ===')
le_disease = LabelEncoder()
y_disease  = le_disease.fit_transform(df_disease['disease'].astype(str))
disease_encoders['disease'] = le_disease

print(f'Классов: {len(le_disease.classes_)}')
model_disease = RandomForestClassifier(**RF_PARAMS)
model_disease = run_cv(model_disease, X_disease, y_disease)

In [ ]:
# --- Classifier 2: уровень риска ---
print('=== RiskClassifier ===')
le_risk = LabelEncoder()
y_risk  = le_risk.fit_transform(df_disease['risk_level'].astype(str))
disease_encoders['risk_level'] = le_risk

print(f'Классы риска: {list(le_risk.classes_)}')
model_risk = RandomForestClassifier(**RF_PARAMS)
model_risk = run_cv(model_risk, X_disease, y_risk)

In [ ]:
# Classification report для обеих моделей
print('--- DiseaseClassifier ---')
print(classification_report(
    y_disease, model_disease.predict(X_disease),
    target_names=le_disease.classes_
))

print('--- RiskClassifier ---')
print(classification_report(
    y_risk, model_risk.predict(X_disease),
    target_names=le_risk.classes_
))

In [ ]:
# Важность признаков для болезни
feature_names = feature_cat + feature_num
disease_imp = pd.Series(
    model_disease.feature_importances_, index=feature_names
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 3))
disease_imp.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Важность признаков (disease model)')
ax.set_xlabel('Feature importance (Gini)')
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(model_disease, MODELS_DIR / 'disease_model.pkl')
joblib.dump(model_risk,    MODELS_DIR / 'disease_risk_model.pkl')
joblib.dump(disease_encoders, MODELS_DIR / 'disease_encoders.pkl')
print('Сохранено: disease_model.pkl, disease_risk_model.pkl, disease_encoders.pkl')

## 6. Итоги и список артефактов

In [ ]:
print('Артефакты в', MODELS_DIR)
print()
for fname in sorted(MODELS_DIR.glob('*.pkl')):
    size_kb = fname.stat().st_size / 1024
    print(f'  {fname.name:45}  {size_kb:>8.1f} KB')

print()
print('Для запуска инференса: agro-ml-service/app/predictors/agro_predictor.py')
print('Для переобучения через CLI: python agro-ml-service/train/train_agro_models.py')

## Пример инференса — демонстрация работы сервиса

In [ ]:
# --- Рекомендация культуры ---
m_crop = joblib.load(MODELS_DIR / 'crop_model.pkl')
le_c   = joblib.load(MODELS_DIR / 'crop_label_encoder.pkl')

# Входные параметры: почва и климат Омской области (ориентировочно)
X_demo = pd.DataFrame([{
    'N': 50, 'P': 35, 'K': 120,
    'temperature': 19.5, 'humidity': 62.0,
    'ph': 6.9, 'rainfall': 210.0,
}])

probs   = m_crop.predict_proba(X_demo)[0]
top_idx = np.argsort(probs)[::-1]

print('Рекомендация культуры (N=50, P=35, K=120, T=19.5, H=62%, pH=6.9, rain=210mm):')
for i in top_idx[:3]:
    print(f'  {le_c.classes_[i]:20} — {probs[i]:.1%}')

In [ ]:
# --- Прогноз болезни ---
m_dis  = joblib.load(MODELS_DIR / 'disease_model.pkl')
m_risk = joblib.load(MODELS_DIR / 'disease_risk_model.pkl')
enc_d  = joblib.load(MODELS_DIR / 'disease_encoders.pkl')

# Входные условия: пшеница, фаза цветения, влажная тёплая погода
crop_enc  = int(enc_d['crop'].transform(['spring_wheat'])[0])
stage_enc = int(enc_d['growth_stage'].transform(['flowering'])[0])

X_dis = np.array([[crop_enc, stage_enc, 22.0, 84.0, 95.0]])

disease_probs = m_dis.predict_proba(X_dis)[0]
top_d  = np.argsort(disease_probs)[::-1]
risk_probs = m_risk.predict_proba(X_dis)[0]
best_risk  = enc_d['risk_level'].classes_[np.argmax(risk_probs)]

print('Прогноз болезни (spring_wheat, flowering, T=22, H=84%, rain=95mm):')
for i in top_d[:3]:
    print(f'  {enc_d["disease"].classes_[i]:35} — {disease_probs[i]:.1%}')
print(f'Уровень риска: {best_risk} (confidence {risk_probs.max():.1%})')